# Module 3 — Multiprocessing

**What it is:** Multiprocessing spawns separate Python processes, each with its own memory and Python interpreter. It bypasses the GIL and is ideal for CPU-bound work like numeric computation and data crunching.

**Where we use it:** Parallel data processing, image rendering, scientific computing, and batch transformations on large datasets.

**Why we use it:** True parallelism on multiple CPU cores speeds up compute-heavy tasks that threading cannot accelerate due to the Global Interpreter Lock (GIL).

In [ ]:
from multiprocessing import Process, cpu_count
import os

def square_numbers(start, end):
    total = sum(x * x for x in range(start, end))
    print(f"PID {os.getpid()}: sum of squares [{start}, {end}) = {total}")

if __name__ == "__main__":
    print("CPU cores available:", cpu_count())
    p1 = Process(target=square_numbers, args=(0, 5_000))
    p2 = Process(target=square_numbers, args=(5_000, 10_000))
    p1.start()
    p2.start()
    p1.join()
    p2.join()

## ProcessPoolExecutor

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

candidates = [17, 18, 19, 20, 97, 100]

if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=2) as pool:
        results = list(pool.map(is_prime, candidates))
    for num, prime in zip(candidates, results):
        print(f"{num}: {'prime' if prime else 'not prime'}")

## Sharing data with Queue

In [ ]:
from multiprocessing import Process, Queue

def producer(queue, items):
    for item in items:
        queue.put(item)
    queue.put(None)  # sentinel

def consumer(queue):
    while True:
        item = queue.get()
        if item is None:
            break
        print("Consumed:", item)

if __name__ == "__main__":
    q = Queue()
    p = Process(target=producer, args=(q, ["apple", "banana", "cherry"]))
    c = Process(target=consumer, args=(q,))
    p.start()
    c.start()
    p.join()
    c.join()